# 2D-to-3D Game Models Pipeline — Test on Colab

**How to use:** Runtime → Change runtime type → **T4 GPU** → Run All

1. First run: installs packages and **restarts runtime automatically**
2. After restart: click **Run All again** — it skips install, runs the pipeline
3. You'll be asked to upload an image, then get a .GLB downloaded to your phone

In [ ]:
#@title 1. Setup (install + clone) — auto-skips on second run
import os, sys, subprocess

REPO_DIR = '/content/2d-to-3d-game-models'
INSTALL_MARKER = '/content/.deps_installed'

def is_installed():
    """Check if packages were already installed (survives kernel restart)."""
    return os.path.exists(INSTALL_MARKER)

if not is_installed():
    print('=== First run: installing packages ===')
    os.chdir('/content')

    # Clone repo
    if os.path.exists(REPO_DIR):
        subprocess.check_call(['rm', '-rf', REPO_DIR])
    subprocess.check_call(['git', 'clone', '-b',
        'claude/image-to-3d-pipeline-CnSII',
        'https://github.com/pmikola/2d-to-3d-game-models.git'])

    # Install scipy compatible with Colab's numpy 2.0.x
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'scipy', 'onnxruntime-gpu'])

    # rembg: use --no-deps + pin version for Colab compat
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--no-deps', 'rembg==2.0.57'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'pooch', 'pymatting', 'scikit-image', 'filetype', 'imagehash'])

    # Everything else
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'trimesh', 'pygltflib', 'xatlas', 'Pillow',
        'diffusers', 'transformers', 'accelerate', 'safetensors',
        'pyyaml', 'tqdm', 'huggingface_hub'])

    # Mark as installed (persists across kernel restart)
    open(INSTALL_MARKER, 'w').write('done')

    print('\n=== Packages installed. Restarting kernel... ===')
    print('>>> After restart, click Run All again <<<\n')

    # Restart kernel (new scipy needs fresh process)
    try:
        import IPython
        IPython.get_ipython().kernel.do_shutdown(True)
    except Exception:
        os._exit(0)

else:
    print('=== Packages already installed, skipping ===')
    os.chdir(REPO_DIR)

    # Quick verification
    import numpy, scipy, trimesh
    print(f'numpy={numpy.__version__}  scipy={scipy.__version__}  trimesh={trimesh.__version__}')

    REMBG_OK = False
    try:
        from rembg import remove
        REMBG_OK = True
        print('rembg: OK')
    except Exception as e:
        print(f'rembg: unavailable ({e}) — will skip background removal')

    import torch
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'GPU: {gpu} ({vram:.1f} GB VRAM)')
    else:
        print('No GPU — running on CPU (slow)')

    print('\n=== Ready! ===')

In [ ]:
#@title 2. Upload your image
from google.colab import files
from PIL import Image
import numpy as np
from IPython.display import display

INPUT_PATH = '/content/test_input.png'

print('Upload a PNG or JPG image from your phone/PC:')
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    import shutil
    shutil.copy(fname, INPUT_PATH)
    img = Image.open(INPUT_PATH)
    print(f'\nUploaded: {fname} ({img.size[0]}x{img.size[1]})')
    display(img.resize((300, 300)))
else:
    # Fallback synthetic image
    print('No upload — using synthetic test image')
    arr = np.zeros((400, 400, 3), dtype=np.uint8)
    arr[:] = np.random.randint(100, 200, (400, 400, 3), dtype=np.uint8)
    y, x = np.ogrid[-200:200, -200:200]
    arr[x**2 + y**2 < 120**2] = [220, 140, 60]
    Image.fromarray(arr).save(INPUT_PATH)
    display(Image.open(INPUT_PATH).resize((200, 200)))

In [ ]:
#@title 3. Run full pipeline: Preprocess → Mesh → UV → PBR → GLB → Download
import os, time, shutil, tempfile, base64
import trimesh
import numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML

import logging
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')

os.chdir('/content/2d-to-3d-game-models')

from pipeline.preprocess import preprocess_image
from pipeline.mesh_repair import repair_and_prepare
from pipeline.geometry import normalize_mesh, unwrap_uvs, save_mesh_as_obj
from pipeline.pbr_maps import generate_pbr_maps, save_pbr_maps
from pipeline.export import export_textured_dir_to_glb, validate_glb

INPUT_PATH = '/content/test_input.png'
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_GLB = str(OUTPUT_DIR / 'model.glb')

start = time.time()

# --- Stage 1: Preprocess ---
print('\n[1/6] Preprocessing image...')
try:
    preprocessed = preprocess_image(INPUT_PATH, target_size=512,
        remove_bg=REMBG_OK, use_gpu=False)
    if REMBG_OK:
        print('  Background removed')
except Exception as e:
    print(f'  Background removal failed ({e}), continuing without it')
    preprocessed = preprocess_image(INPUT_PATH, target_size=512,
        remove_bg=False, use_gpu=False)
print(f'  Result: {preprocessed.size}')
display(preprocessed.resize((200, 200)))

# --- Stage 2: Generate mesh (synthetic — real models need Hi3DGen/Hunyuan3D) ---
print('\n[2/6] Generating mesh (synthetic demo — real pipeline uses Hi3DGen)...')
mesh = trimesh.creation.icosphere(subdivisions=4, radius=1.0)
mesh.vertices += np.random.normal(0, 0.01, mesh.vertices.shape)
print(f'  Vertices: {len(mesh.vertices)}, Faces: {len(mesh.faces)}')

# --- Stage 3: Mesh repair ---
print('\n[3/6] Repairing mesh (topology fix, smoothing)...')
mesh = repair_and_prepare(mesh, smooth_iterations=3)
print(f'  After repair: {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

# --- Stage 4: Normalize + UV unwrap ---
print('\n[4/6] UV unwrapping (xatlas)...')
normalize_mesh(mesh)
unwrap_uvs(mesh)
print(f'  UV unwrapped: {len(mesh.vertices)} verts')

# --- Stage 5: PBR texture maps ---
print('\n[5/6] Generating PBR maps (normal / roughness / metallic)...')
with tempfile.TemporaryDirectory() as tmp:
    obj_path = save_mesh_as_obj(mesh, tmp)
    textured = Path(tmp) / 'textured'
    textured.mkdir()

    preprocessed.save(str(textured / 'texture_atlas.png'))
    shutil.copy(obj_path, str(textured / 'mesh_textured.obj'))

    pbr = generate_pbr_maps(preprocessed, strength=1.5)
    save_pbr_maps(pbr, str(textured / 'pbr'))

    row = Image.new('RGB', (512*4, 512))
    row.paste(preprocessed.resize((512,512)), (0,0))
    row.paste(pbr['normal'].resize((512,512)), (512,0))
    row.paste(pbr['roughness'].convert('RGB').resize((512,512)), (1024,0))
    row.paste(pbr['metallic'].convert('RGB').resize((512,512)), (1536,0))
    print('  Albedo | Normal | Roughness | Metallic:')
    display(row.resize((800, 200)))

    # --- Stage 6: Export GLB ---
    print('\n[6/6] Exporting GLB with embedded PBR textures...')
    export_textured_dir_to_glb(str(textured), OUTPUT_GLB)

# Validate
info = validate_glb(OUTPUT_GLB)
elapsed = time.time() - start
size_mb = os.path.getsize(OUTPUT_GLB) / (1024*1024)

print(f'\n{"="*50}')
print(f'DONE in {elapsed:.1f}s')
print(f'  GLB: {OUTPUT_GLB} ({size_mb:.2f} MB)')
print(f'  {info}')
print(f'{"="*50}')

# === DOWNLOAD ===
# Always create a clickable download link (works on any device)
print('\n')
with open(OUTPUT_GLB, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()
display(HTML(
    f'<h2><a href="data:model/gltf-binary;base64,{b64}" '
    f'download="model.glb" '
    f'style="background:#4CAF50;color:white;padding:15px 30px;'
    f'text-decoration:none;border-radius:8px;font-size:18px;">'
    f'📥 TAP HERE TO DOWNLOAD model.glb ({size_mb:.1f} MB)</a></h2>'
))
print('\nAfter downloading, view your model at:')
print('https://gltf-viewer.donmccurdy.com/')